# Use the RDF Data of the Swiss Political Parties

In [1]:
# load graph from the Turtle file
from rdflib import Graph

g = Graph()
g.parse("../data/output/parties.ttl", format="turtle")

<Graph identifier=N6e3d3d8e5e2a4972ad94dbc88186c8c5 (<class 'rdflib.graph.Graph'>)>

In [2]:
import pandas as pd

def sparql_to_df(graph: Graph, query: str) -> pd.DataFrame:
    result = graph.query(query)
    return pd.DataFrame(
        [
            [val.toPython() if val is not None else None for val in row]
            for row in result
        ],
        columns=[str(var) for var in result.vars]
    )

## Number of Versions per Party

In [3]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?party (COUNT(?version) AS ?count) WHERE {
    ?party a vl:Identity .
    ?version vl:identity ?party .
} GROUP BY ?party ORDER BY DESC(?count)

"""

In [4]:
df = sparql_to_df(g, query)

display(df)

,party,count
0,https://politics.ld.admin.ch/party-id/128,11
1,https://politics.ld.admin.ch/party-id/236,8
2,https://politics.ld.admin.ch/party-id/311,8
3,https://politics.ld.admin.ch/party-id/37,8
4,https://politics.ld.admin.ch/party-id/103,7
...,...,...
301,https://politics.ld.admin.ch/party-id/85,1
302,https://politics.ld.admin.ch/party-id/90,1
303,https://politics.ld.admin.ch/party-id/95,1
304,https://politics.ld.admin.ch/party-id/98,1


## Identities without Versions

Should not happen

In [5]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id vl:version ?version .
    }
}
    
"""

In [6]:
df = sparql_to_df(g, query)

display(df)

,id
0,https://politics.ld.admin.ch/party-id/64
1,https://politics.ld.admin.ch/party-id/93


## All Relationships of a Party Version

In [7]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?version ?from ?name WHERE {
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?version .

    ?version schema:validFrom ?from .

    OPTIONAL {
        ?version schema:name ?name .
        FILTER (lang(?name) = "de")
    }

} ORDER BY ?from
    
"""

In [8]:
df = sparql_to_df(g, query)

display(df)

,version,from,name
0,https://politics.ld.admin.ch/party-version/16_...,1880-01-01,Vorläufer Demokraten
1,https://politics.ld.admin.ch/party-version/17_...,1905-03-19,Schweizerische Demokratische Partei
2,https://politics.ld.admin.ch/party-version/16_...,1905-03-19,NaN
3,https://politics.ld.admin.ch/party-version/10_...,1912-04-22,Schweizerische Konservative Volkspartei
4,https://politics.ld.admin.ch/party-version/209...,1920-01-01,"Vorläufer Bauern-, Gewerbe und Bürgerpartei"
5,https://politics.ld.admin.ch/party-version/11_...,1936-12-23,"Bauern-, Gewerbe- und Bürgerpartei"
6,https://politics.ld.admin.ch/party-version/209...,1936-12-23,NaN
7,https://politics.ld.admin.ch/party-version/18_...,1941-07-01,Demokratische Partei der Schweiz
8,https://politics.ld.admin.ch/party-version/17_...,1941-07-01,NaN
9,https://politics.ld.admin.ch/party-version/10_...,1957-07-01,Konservativ-Christlichsoziale Volkspartei


## Visualization of the Relationships of a Party Version

In [9]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target WHERE {
    
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?source .

    ?source vl:successor ?target

}
    
"""

In [10]:
df = sparql_to_df(g, query)

display(df)

,source,target
0,https://politics.ld.admin.ch/party-version/13_...,https://politics.ld.admin.ch/party-version/13_...
1,https://politics.ld.admin.ch/party-version/13_...,https://politics.ld.admin.ch/party-version/4_2...
2,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...
3,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/4_2...
4,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...
5,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/66_...
6,https://politics.ld.admin.ch/party-version/66_...,https://politics.ld.admin.ch/party-version/66_...
7,https://politics.ld.admin.ch/party-version/65_...,https://politics.ld.admin.ch/party-version/65_...
8,https://politics.ld.admin.ch/party-version/65_...,https://politics.ld.admin.ch/party-version/66_...
9,https://politics.ld.admin.ch/party-version/10_...,https://politics.ld.admin.ch/party-version/10_...


In [11]:
from ipycytoscape import CytoscapeWidget
import networkx as nx

my_style = [
    {
        "selector": "node",
        "style": {
            "label": "data(id)",
            "background-color": "#0074D9",
            "color": "#fff",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": 10,
            "width": 20,
            "height": 20,
        },
    },
    {
        "selector": "edge.directed",
        "style": {
            "line-color": "#0074D9",
            "target-arrow-color": "#0074D9",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
        },
    },
]

G = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())
cyto = CytoscapeWidget()
cyto.graph.add_graph_from_networkx(G, directed=True)
display(cyto)

CytoscapeWidget(cytoscape_layout={'name': 'cola'}, cytoscape_style=[{'selector': 'node', 'css': {'background-c…